In [13]:
"""
Faithfulness-Optimized RAG
===========================
Primary References:
  FActScore  : Min et al., "FActScore: Fine-Grained Atomic Evaluation of
               Factual Precision in Long Form Text Generation"
               arXiv: 2305.14251 (EMNLP 2023)
  RAGAS      : Es et al., "RAGAS: Automated Evaluation of Retrieval
               Augmented Generation"
               arXiv: 2309.15217 (EACL 2024)
  HHEM       : Hughes Hallucination Evaluation Model, Vectara 2023-2025
               "Benchmarking LLM Faithfulness in RAG with Evolving Leaderboards"
               arXiv: 2505.04847 (Nov 2025, FaithJudge)
  FRANQ      : "Faithfulness-Aware Uncertainty Quantification for Fact-Checking
               the Output of Retrieval Augmented Generation"
               arXiv: 2505.21072 (May 2025)
  Self-RAG   : Asai et al., "Self-RAG: Learning to Retrieve, Generate, and
               Critique through Self-Reflection"
               arXiv: 2310.11511 (NeurIPS 2023)

Architecture: FIVE configurations covering the full faithfulness design space:
    (1) Unchecked Baseline     -- standard RAG: retrieve -> generate, no
                                  faithfulness measurement or enforcement.
                                  This is the CONTROL condition: it establishes
                                  the hallucination rate of unconstrained RAG.
    (2) FActScore Audit        -- post-generation atomic claim decomposition +
                                  NLI entailment check per claim against context.
                                  MEASURE faithfulness but do NOT modify generation.
                                  Score = |supported claims| / |total claims|.
    (3) RAGAS Faithfulness     -- RAGAS-style pipeline:
                                  (a) LLM decomposes answer into atomic statements,
                                  (b) LLM verifies each statement against context,
                                  (c) faithfulness = supported / total statements.
                                  Extended with citation injection: each sentence
                                  in the answer is annotated with [Source: ...].
    (4) Citation-Grounded RAG  -- generation is CONSTRAINED to cite sources.
                                  The generation prompt requires the model to produce
                                  inline citations for every factual claim.
                                  Post-generation: citation accuracy is verified by
                                  checking whether the cited passage actually supports
                                  the claim (NLI entailment or LLM-as-judge).
    (5) Faithful-RAG with      -- full production pipeline: generate -> decompose
        Regeneration [BEST]       claims -> check each claim -> if any claim is
                                  UNSUPPORTED or CONTRADICTED: trigger targeted
                                  regeneration of ONLY the unfaithful sentences,
                                  using the specific supporting evidence.
                                  Includes a faithfulness score threshold gate:
                                  answers below FAITHFULNESS_THRESHOLD are fully
                                  regenerated with a stricter "only-from-context" prompt.

=============================================================================
THE FAITHFULNESS PROBLEM: WHY RAG DOES NOT AUTOMATICALLY PREVENT HALLUCINATION
=============================================================================
A widely held assumption is that RAG eliminates hallucination by grounding
generation in retrieved documents. This assumption is INCORRECT in two ways.

First, even when relevant context is provided, LLMs still introduce unsupported
information. The Vectara HHEM leaderboard (2023-2025) consistently shows
hallucination rates of 3-27% across modern LLMs on RAG summarization tasks,
depending on model family. RAG aims to reduce hallucinations by grounding
responses in external context, yet LLMs still frequently introduce unsupported
information or contradictions even when provided with relevant context.

Second, and more subtly, the model's parametric knowledge can CONFLICT with
the retrieved context. The internal knowledge of the LLM and the retrieved
information may contradict each other. The retrieved information may even be
erroneous, incomplete, or completely irrelevant to the query. When this happens,
the model may silently blend parametric and contextual knowledge, producing
answers that are partially faithful and partially hallucinated without any
visible signal of the blend.

The taxonomy of faithfulness failures (Huang et al., 2024):
    Faithful hallucination:  the generated claim is NOT supported by context
                             but may or may not be factually correct in the world.
                             Example: model states a number not in any retrieved chunk.
    Contradiction:           the generated claim DIRECTLY CONTRADICTS context.
                             Example: context says "512 dimensions", model says "768".
    Extrinsic injection:     the model adds information from parametric memory
                             that is not in the context ("additional" background).

FActScore, RAGAS faithfulness, and HHEM/FaithJudge all measure FAITHFULNESS
(alignment with retrieved context), NOT FACTUALITY (alignment with the world).
A faithful answer is grounded in the context whether or not the context is correct.
This distinction matters for production: a faithful answer to a wrong context
is still hallucination in the world-knowledge sense.

=============================================================================
CLAIM DECOMPOSITION: THE FOUNDATION OF ALL FAITHFULNESS METRICS
=============================================================================
FActScore's key insight: evaluating faithfulness at the ANSWER level is too coarse.
A long answer may contain 10 claims: 8 supported, 2 unsupported.
An answer-level binary (faithful/unfaithful) labels this answer as "unfaithful"
and loses the signal that 80% of the claims are correct.

FActScore decomposes long-form answers into ATOMIC CLAIMS: each claim is a
single, independently verifiable factual statement. Then faithfulness is scored
as the PROPORTION of atomic claims that are supported by evidence.
FActScore is a dataset-agnostic scoring framework that measures the proportion
of atomic facts supported by evidence; it underlies many subsequent metrics.

The RAGAS implementation uses the same principle:
The Faithfulness metric measures how factually consistent a response is with
the retrieved context. It ranges from 0 to 1, with higher scores indicating
better consistency. A response is considered faithful if all its claims can
be supported by the retrieved context.

Two-stage RAGAS faithfulness pipeline:
    Stage 1: LLM decomposes the answer into a list of atomic statements.
    Stage 2: LLM checks each statement against the retrieved context.
    Score = |statements that can be inferred from context| / |total statements|

To operationalize faithfulness, NLI pipelines decompose responses into atomic
claims, then classify each as entailed, neutral, or contradicted by the context
using models like DeBERTa. Aggregation offers a faithfulness score as a
proportion of entailed claims, with thresholds above 0.9 indicating grounded
outputs.

=============================================================================
CITATION GROUNDING: FROM MEASUREMENT TO ENFORCEMENT
=============================================================================
FActScore and RAGAS MEASURE faithfulness after generation.
Citation-Grounded RAG ENFORCES faithfulness DURING generation by requiring
the model to cite the specific passage that supports each factual claim.

This turns faithfulness from a post-hoc audit into a generation constraint:
    - The model cannot generate an unsupported claim without a citation.
    - Every [Source: X, p.N] is a verifiable pointer to the supporting evidence.
    - Post-generation verification checks whether the cited passage actually
      entails the claim (citation accuracy, distinct from answer faithfulness).

The dual verification (answer faithfulness + citation accuracy) is the
production standard for high-stakes RAG (legal, medical, financial).

=============================================================================
FAITHFUL-RAG WITH REGENERATION: CLOSING THE FEEDBACK LOOP
=============================================================================
The full Faithful-RAG pipeline (Config 5) closes the feedback loop:
    Generate -> Decompose -> Check each claim -> Identify unfaithful claims
    -> Targeted regeneration: for each unfaithful claim, retrieve the most
       relevant evidence and regenerate ONLY that claim.
    -> If total faithfulness < FAITHFULNESS_THRESHOLD: full regeneration
       with a stricter "answer ONLY from the following passages" prompt.

This is the production realization of Self-RAG's CRITIQUE tokens (Asai et al.,
2023): "ISREL" (is retrieved passage relevant?), "ISSUP" (is the claim supported?),
"ISUSE" (is the answer useful?). Config 5 implements the core ISSUP loop without
requiring a fine-tuned model: it uses an LLM-as-judge for claim verification
and targeted regeneration for repair.
"""

'\nFaithfulness-Optimized RAG\n===========================\nPrimary References:\n  FActScore  : Min et al., "FActScore: Fine-Grained Atomic Evaluation of\n               Factual Precision in Long Form Text Generation"\n               arXiv: 2305.14251 (EMNLP 2023)\n  RAGAS      : Es et al., "RAGAS: Automated Evaluation of Retrieval\n               Augmented Generation"\n               arXiv: 2309.15217 (EACL 2024)\n  HHEM       : Hughes Hallucination Evaluation Model, Vectara 2023-2025\n               "Benchmarking LLM Faithfulness in RAG with Evolving Leaderboards"\n               arXiv: 2505.04847 (Nov 2025, FaithJudge)\n  FRANQ      : "Faithfulness-Aware Uncertainty Quantification for Fact-Checking\n               the Output of Retrieval Augmented Generation"\n               arXiv: 2505.21072 (May 2025)\n  Self-RAG   : Asai et al., "Self-RAG: Learning to Retrieve, Generate, and\n               Critique through Self-Reflection"\n               arXiv: 2310.11511 (NeurIPS 2023)\n\nArch

In [14]:

import os
import re
import sys
import json
import time
import logging
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any


In [15]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [16]:

# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("faithfulness_rag")



In [17]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class FaithfulRAGConfig:
    """
    Centralized configuration for the Faithfulness-Optimized RAG pipeline.

    FAITHFULNESS_THRESHOLD (0.80):
        Minimum proportion of supported atomic claims required for an answer
        to be considered acceptably faithful. Answers below this threshold
        in Config 5 are flagged for full regeneration.
        Source: RAGAS documentation notes 0.9 as "grounded" threshold;
        0.80 is a pragmatic production threshold balancing strictness
        and answer completeness.
        At 0.80: at most 20% of claims can be unsupported before regeneration.
        At 0.95: extremely strict, forces regeneration for minor LLM interpolations.

    MAX_CLAIMS_PER_ANSWER (12):
        Maximum number of atomic claims extracted per answer in Configs 2-5.
        LLM-based claim decomposition can generate 15-30 claims for a long answer.
        Capping at 12 keeps the verification cost predictable: each claim
        requires 1 LLM verification call, so 12 claims = 12 additional LLM calls
        per answer in the NLI-as-LLM verification path.

    CITATION_REQUIRED (True):
        In Config 4 (Citation-Grounded), whether to REQUIRE inline citations
        for every factual claim. When True: the generation prompt explicitly
        instructs the model to append [Source: paper_name, p.N] after each
        claim. When False: citations are encouraged but not enforced.
        Production default: True for high-stakes domains.

    REGENERATION_GRANULARITY ("sentence"):
        In Config 5 (Faithful-RAG + Regeneration), the unit at which
        unfaithful content is regenerated:
            "sentence": regenerate only the specific sentence containing
                        the unsupported claim. Preserves supported sentences.
            "full":     regenerate the entire answer from scratch with
                        a stricter prompt. Used when faithfulness < threshold.

    NLI_ENTAILMENT_METHOD ("llm"):
        The method used to verify whether a claim is entailed by context:
            "llm":      LLM-as-judge: the LLM reads the claim and context
                        and outputs SUPPORTED/UNSUPPORTED/CONTRADICTED.
                        Accuracy: ~75-78% balanced accuracy (FaithBench 2025).
            "nli":      NLI model (e.g., DeBERTa-v3-large-mnli).
                        Faster, cheaper, no LLM calls for verification.
                        Note: embedding-based NLI fails on real hallucinations
                        (100% FPR at 95% recall per arXiv 2512.15068).
            "hybrid":   NLI for clear cases (confidence > 0.9), LLM for ambiguous.
        Default "llm": the LLM judge outperforms embedding NLI on real
        hallucinations and is the method validated in FaithBench 2025.
    """

    # --- Dataset ----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"
    FAISS_DIR:        str = "./faiss_faithful_rag"

    # --- BGE Embeddings ---------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Chunking ---------------------------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- Retrieval --------------------------------------------------------
    TOP_K_RETRIEVAL: int = 5
    TOP_K_FINAL:     int = 4

    # --- Faithfulness Parameters ------------------------------------------
    FAITHFULNESS_THRESHOLD:       float = 0.80
    MAX_CLAIMS_PER_ANSWER:        int   = 12
    CITATION_REQUIRED:            bool  = True
    REGENERATION_GRANULARITY:     str   = "sentence"   # "sentence" | "full"
    NLI_ENTAILMENT_METHOD:        str   = "llm"        # "llm" | "nli" | "hybrid"
    UNSUPPORTED_LABEL:            str   = "UNSUPPORTED"
    SUPPORTED_LABEL:              str   = "SUPPORTED"
    CONTRADICTED_LABEL:           str   = "CONTRADICTED"

    # --- LLM --------------------------------------------------------------
    LLM_TEMPERATURE:         float = 0.0
    AI_FOUNDRY_PROJECT_ENDPOINT: str = "https://idkopenai.services.ai.azure.com"
    AI_FOUNDRY_DEPLOYMENT_NAME: str = "gpt-4o"
    AI_FOUNDRY_API_VERSION: str = "2024-12-01-preview"
    AI_FOUNDRY_API_KEY: str = ""
    LLM_MAX_TOKENS:          int   = 1024

    # --- Context ----------------------------------------------------------
    MAX_CONTEXT_CHARS: int = 3200

    # ======================================================================
    # PROMPTS
    # ======================================================================

    # Standard generation prompt (Config 1 and 2 -- no citation requirement)
    P_GENERATE_STANDARD: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, say: "The provided documents do not contain sufficient information."

Context:
{context}

Question: {question}

Answer:"""

    # Citation-enforcing generation prompt (Config 3, 4, 5)
    P_GENERATE_WITH_CITATIONS: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
For every factual claim in your answer, you MUST append a citation in the format:
[Source: <paper_name>, p.<page>]

Rules:
1. Every sentence containing a factual claim MUST have a citation.
2. Do NOT invent information not present in the context.
3. If the answer is not present, say: "The provided documents do not contain sufficient information."
4. Use the exact paper name and page number from the context headers.

Context:
{context}

Question: {question}

Answer (with inline citations):"""

    # Strict "only-from-context" regeneration prompt (Config 5 fallback)
    P_REGENERATE_STRICT: str = """You are a precise technical research assistant.
You MUST answer STRICTLY from the following passages. Every word of your answer
must be directly traceable to one of the passages below.
Do NOT add background knowledge, transitions, or explanations not in the passages.
Append [Source: <paper_name>, p.<page>] after every sentence.

Passages:
{context}

Question: {question}

Strictly context-grounded answer:"""

    # Targeted sentence regeneration (Config 5 -- fix one specific sentence)
    P_REGENERATE_SENTENCE: str = """The following sentence in an answer is NOT supported by evidence.
Rewrite it to be FAITHFUL to the provided supporting passage.
If the passage does not support the claim, replace the sentence with: [Not supported -- removed.]

Original sentence: {original_sentence}
Supporting passage: {supporting_passage}
Claim that is unsupported: {unsupported_claim}

Rewritten sentence (faithful to passage):"""

    # FActScore / RAGAS: decompose answer into atomic claims
    P_DECOMPOSE_CLAIMS: str = """Decompose the following answer into a list of ATOMIC CLAIMS.
Each atomic claim must be:
  - A single, independently verifiable factual statement.
  - Self-contained (can be understood without the rest of the answer).
  - Not a judgment or opinion -- only factual assertions.

Output ONLY a JSON list of strings, one claim per string. No preamble.
Example output: ["Claim 1.", "Claim 2.", "Claim 3."]

Answer to decompose:
{answer}

JSON list of atomic claims:"""

    # LLM-as-judge NLI verification (per claim)
    P_VERIFY_CLAIM: str = """Determine if the following claim is supported by the context passages.
Output EXACTLY one word: SUPPORTED, UNSUPPORTED, or CONTRADICTED.

SUPPORTED   -- the claim can be directly inferred from the context.
UNSUPPORTED -- the claim is not mentioned or cannot be inferred from the context.
CONTRADICTED-- the claim directly contradicts something stated in the context.

Context:
{context}

Claim to verify: {claim}

Verdict (one word only):"""

    # Citation accuracy check (Config 4)
    P_VERIFY_CITATION: str = """Does the following passage support the claim?
The passage is the cited source for the claim. Output YES or NO only.

Claim: {claim}
Cited passage: {passage}

Answer (YES or NO):"""



In [18]:

# ===========================================================================
# SECTION 2 -- DATA STRUCTURES
# ===========================================================================

@dataclass
class ClaimVerification:
    """
    Verification result for one atomic claim.

    claim:      the atomic claim string extracted from the answer.
    verdict:    SUPPORTED / UNSUPPORTED / CONTRADICTED.
    evidence:   the passage that was used for verification (top-1 retrieved).
    llm_ms:     latency of the LLM verification call.
    confidence: optional probability from an NLI model [0, 1].
    """
    claim:      str
    verdict:    str
    evidence:   str   = ""
    llm_ms:     float = 0.0
    confidence: float = 1.0

    @property
    def is_supported(self) -> bool:
        return self.verdict.upper().startswith("SUPPORTED")

    @property
    def is_contradicted(self) -> bool:
        return self.verdict.upper().startswith("CONTRADICTED")



In [19]:


@dataclass
class CitationVerification:
    """Verification result for one inline citation in the answer."""
    sentence:      str
    citation_ref:  str    # e.g., "Attention Is All You Need, p.3"
    cited_passage: str    # the actual retrieved passage text
    is_accurate:   bool   # does the passage support the claim?
    llm_ms:        float  = 0.0


@dataclass
class FaithfulnessReport:
    """
    Complete faithfulness audit for one generated answer.

    faithfulness_score: |supported_claims| / |total_claims|  in [0, 1].
    claims:             list of ClaimVerification objects.
    citations:          list of CitationVerification objects (Config 4+).
    supported_count:    number of supported claims.
    unsupported_count:  number of unsupported claims.
    contradicted_count: number of contradicted claims.
    regenerated:        True if the answer was regenerated due to low faithfulness.
    final_answer:       the (potentially regenerated) answer after faithful optimization.
    audit_llm_calls:    LLM calls used for faithfulness auditing only.
    audit_ms:           total time for the audit phase.
    """
    faithfulness_score:  float
    claims:              List[ClaimVerification] = field(default_factory=list)
    citations:           List[CitationVerification] = field(default_factory=list)
    supported_count:     int   = 0
    unsupported_count:   int   = 0
    contradicted_count:  int   = 0
    regenerated:         bool  = False
    final_answer:        str   = ""
    audit_llm_calls:     int   = 0
    audit_ms:            float = 0.0

    def print_report(self) -> None:
        bar = "=" * 68
        print(f"\n{bar}")
        print(f"FAITHFULNESS REPORT")
        print(f"  Score:        {self.faithfulness_score:.3f} "
              f"({self.supported_count} supported / "
              f"{self.supported_count + self.unsupported_count + self.contradicted_count} total claims)")
        print(f"  Unsupported:  {self.unsupported_count}")
        print(f"  Contradicted: {self.contradicted_count}")
        print(f"  Regenerated:  {self.regenerated}")
        print(f"  Audit calls:  {self.audit_llm_calls}  |  Audit ms: {self.audit_ms:.0f}")
        print(f"\n  Claims:")
        for cv in self.claims:
            icon = "OK" if cv.is_supported else ("XX" if cv.is_contradicted else "--")
            print(f"  [{icon}] {cv.verdict:<14} | {cv.claim[:65]}")
        if self.citations:
            print(f"\n  Citations:")
            for cit in self.citations:
                acc_tag = "ACCURATE" if cit.is_accurate else "INACCURATE"
                print(f"  [{acc_tag}] [{cit.citation_ref[:40]}]  "
                      f"-> '{cit.sentence[:50]}'")
        print(bar + "\n")


@dataclass
class FaithfulRAGTrace:
    """Full trace for one question across generation and faithfulness auditing."""
    question:        str
    strategy:        str
    raw_answer:      str   = ""
    final_answer:    str   = ""
    context_str:     str   = ""
    faithfulness:    Optional[FaithfulnessReport] = None
    generation_llm:  int   = 0
    total_llm_calls: int   = 0
    retrieval_ms:    float = 0.0
    generation_ms:   float = 0.0
    audit_ms:        float = 0.0
    total_ms:        float = 0.0

    def print_trace(self) -> None:
        print(f"\n{'='*72}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:72]}")
        print(f"{'='*72}")
        if self.faithfulness:
            self.faithfulness.print_report()
        print(f"  LLM calls:   {self.total_llm_calls}  "
              f"| Retrieval: {self.retrieval_ms:.0f}ms  "
              f"| Gen: {self.generation_ms:.0f}ms  "
              f"| Audit: {self.audit_ms:.0f}ms  "
              f"| Total: {self.total_ms:.0f}ms")
        print(f"\n  FINAL ANSWER:\n  {self.final_answer[:480]}")
        print("=" * 72 + "\n")


In [20]:


# ===========================================================================
# SECTION 3 -- INFRASTRUCTURE
# ===========================================================================

def download_pdfs(config: FaithfulRAGConfig) -> List[Path]:
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "FaithfulRAG/1.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            dest.write_bytes(data)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(f"Cannot download '{url}': {exc}") from exc
    return paths


def load_and_chunk(pdf_paths: List[Path], config: FaithfulRAGConfig) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""], add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        pages      = PyPDFLoader(str(pdf_path)).load()
        for page in pages:
            page.metadata["paper_name"] = paper_name
            page.metadata["source"]     = pdf_path.name
        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d chunks", pdf_path.name, len(chunks))
        all_chunks.extend(chunks)
    return all_chunks


def build_bge_embeddings(config: FaithfulRAGConfig) -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss(chunks: List[Document], embeddings: HuggingFaceEmbeddings,
                config: FaithfulRAGConfig) -> FAISS:
    faiss_file = Path(config.FAISS_DIR) / "index.faiss"
    if faiss_file.exists():
        try:
            vs = FAISS.load_local(config.FAISS_DIR, embeddings,
                                  allow_dangerous_deserialization=True)
            return vs
        except Exception:
            pass
    vs = FAISS.from_documents(chunks, embeddings)
    Path(config.FAISS_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_DIR)
    return vs


def build_bm25(chunks: List[Document]) -> BM25Retriever:
    bm25   = BM25Retriever.from_documents(chunks)
    bm25.k = 5
    return bm25


def build_azure_llm(config: FaithfulRAGConfig) -> AzureChatOpenAI:
    """Connect to Azure OpenAI and return an AzureChatOpenAI instance."""
    return AzureChatOpenAI(
        azure_endpoint=getattr(config, 'AI_FOUNDRY_PROJECT_ENDPOINT', 'https://idkopenai.services.ai.azure.com'),
        azure_deployment=getattr(config, 'AI_FOUNDRY_DEPLOYMENT_NAME', 'gpt-4o'),
        api_version=getattr(config, 'AI_FOUNDRY_API_VERSION', '2024-12-01-preview'),
        api_key=getattr(config, 'AI_FOUNDRY_API_KEY', ''),
        temperature=getattr(config, 'LLM_TEMPERATURE', 0.1),
        max_tokens=getattr(config, 'LLM_MAX_TOKENS', 512),
    )
def llm_call(prompt: str, llm: AzureChatOpenAI) -> str:
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()


def retrieve_hybrid(question: str, vs: FAISS, bm25: BM25Retriever,
                    config: FaithfulRAGConfig) -> Tuple[List[Document], float]:
    """Hybrid dense + sparse retrieval with RRF fusion. Returns (docs, retrieval_ms)."""
    t0          = time.perf_counter()
    dense_docs  = vs.similarity_search(question, k=config.TOP_K_RETRIEVAL)
    sparse_docs = bm25.invoke(question)
    rrf_scores: Dict[int, float]    = {}
    doc_map:    Dict[int, Document] = {}
    for rank, doc in enumerate(dense_docs):
        h = hash(doc.page_content)
        rrf_scores[h] = rrf_scores.get(h, 0.0) + 1.0 / (60 + rank + 1)
        doc_map[h] = doc
    for rank, doc in enumerate(sparse_docs):
        h = hash(doc.page_content)
        rrf_scores[h] = rrf_scores.get(h, 0.0) + 1.0 / (60 + rank + 1)
        doc_map[h] = doc
    sorted_hashes = sorted(rrf_scores, key=lambda h: rrf_scores[h], reverse=True)
    docs = [doc_map[h] for h in sorted_hashes[:config.TOP_K_FINAL]]
    ms   = (time.perf_counter() - t0) * 1000
    return docs, ms


def format_context(docs: List[Document], max_chars: int = 3200) -> str:
    """Format retrieved documents into a context string with paper/page headers."""
    parts, total = [], 0
    for doc in docs:
        paper = doc.metadata.get("paper_name", "Unknown")[:24]
        page  = doc.metadata.get("page", "?")
        part  = f"[{paper} p{page}]\n{doc.page_content.strip()}"
        if total + len(part) > max_chars:
            remaining = max_chars - total
            if remaining > 80:
                parts.append(part[:remaining])
            break
        parts.append(part)
        total += len(part)
    return "\n\n---\n\n".join(parts)


# ===========================================================================
# SECTION 4 -- FAITHFULNESS PRIMITIVES
# ===========================================================================

def decompose_claims(
    answer: str, llm: AzureChatOpenAI, config: FaithfulRAGConfig
) -> Tuple[List[str], float]:
    """
    FActScore / RAGAS Stage 1: Decompose an answer into atomic claims.

    Uses an LLM to extract a JSON list of atomic, independently verifiable
    factual statements from the answer. Each claim is a single sentence
    containing exactly ONE verifiable assertion.

    Returns:
        (list_of_claim_strings, llm_latency_ms)

    Error handling: if JSON parsing fails, falls back to sentence splitting.
    Sentence splitting is a reliable fallback because most LLM answers are
    already sentence-structured, and one sentence typically contains one claim.
    """
    t0     = time.perf_counter()
    prompt = config.P_DECOMPOSE_CLAIMS.format(answer=answer)
    raw    = llm_call(prompt, llm)
    ms     = (time.perf_counter() - t0) * 1000

    # Attempt JSON parse
    try:
        # Strip markdown code fences if present
        cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", raw).strip()
        claims  = json.loads(cleaned)
        if isinstance(claims, list):
            return [str(c).strip() for c in claims[:config.MAX_CLAIMS_PER_ANSWER]], ms
    except (json.JSONDecodeError, ValueError):
        pass

    # Fallback: split on sentence boundaries
    sentences = re.split(r"(?<=[.!?])\s+", answer.strip())
    claims    = [s.strip() for s in sentences if len(s.strip()) > 10]
    return claims[:config.MAX_CLAIMS_PER_ANSWER], ms


def verify_claim_llm(
    claim: str, context: str, llm: AzureChatOpenAI, config: FaithfulRAGConfig
) -> Tuple[str, float]:
    """
    LLM-as-judge NLI entailment verification for one atomic claim.

    Returns (verdict_string, llm_ms) where verdict is one of:
        SUPPORTED, UNSUPPORTED, CONTRADICTED.

    The LLM judge outperforms embedding-based NLI on real hallucinations
    per the FaithBench 2025 leaderboard (Vectara, arXiv 2505.04847).
    Embedding NLI achieves 0% FPR on SYNTHETIC data but 100% FPR on REAL
    hallucinations (arXiv 2512.15068 -- the Semantic Illusion paper).
    The LLM judge achieves ~7% FPR at 95% recall on real hallucinations.
    """
    t0     = time.perf_counter()
    prompt = config.P_VERIFY_CLAIM.format(
        context=context[:1800],  # cap context for verification call
        claim=claim,
    )
    raw_verdict = llm_call(prompt, llm)
    ms          = (time.perf_counter() - t0) * 1000

    # Normalize to one of the three labels
    raw_up = raw_verdict.strip().upper()
    if raw_up.startswith("CONTRADICTED"):
        verdict = config.CONTRADICTED_LABEL
    elif raw_up.startswith("SUPPORTED"):
        verdict = config.SUPPORTED_LABEL
    else:
        verdict = config.UNSUPPORTED_LABEL
    return verdict, ms


def compute_faithfulness_score(verifications: List[ClaimVerification]) -> float:
    """
    Compute RAGAS-style faithfulness score:
        score = |SUPPORTED claims| / |total claims|

    Contradicted claims are counted as unsupported (worst case for the answer).
    """
    if not verifications:
        return 1.0
    supported = sum(1 for cv in verifications if cv.is_supported)
    return supported / len(verifications)


def extract_inline_citations(answer: str) -> List[Tuple[str, str]]:
    """
    Parse inline citations from a generated answer in the format:
        [Source: paper_name, p.N]

    Returns a list of (sentence_containing_citation, citation_ref_string).
    Used by Config 4 to verify citation accuracy.
    """
    citation_pattern = re.compile(
        r"([^.!?]+[.!?])\s*\[Source:\s*([^\]]+)\]",
        re.IGNORECASE,
    )
    matches = citation_pattern.findall(answer)
    return [(m[0].strip(), m[1].strip()) for m in matches]



In [21]:

# ===========================================================================
# SECTION 5 -- FAITHFULNESS AUDIT ENGINE
# ===========================================================================

def run_factscore_audit(
    answer: str, context: str, docs: List[Document],
    llm: AzureChatOpenAI, vs: FAISS, config: FaithfulRAGConfig,
) -> FaithfulnessReport:
    """
    FActScore-style faithfulness audit:
        1. Decompose answer into atomic claims (1 LLM call).
        2. For each claim: retrieve most relevant passage (FAISS lookup).
        3. Verify each claim against its retrieved evidence (1 LLM call each).
        4. Compute faithfulness_score = supported / total.

    Note: FActScore uses a SEPARATE per-claim retrieval (claim is used as
    the retrieval query) rather than the full context. This is more precise
    than checking each claim against the entire context because a long context
    may bury the relevant sentence and the LLM judge may miss it.
    """
    t0            = time.perf_counter()
    audit_calls   = 0

    # Stage 1: Decompose answer into atomic claims
    claims, decompose_ms = decompose_claims(answer, llm, config)
    audit_calls += 1
    logger.info("FActScore audit: %d claims extracted.", len(claims))

    # Stage 2+3: Per-claim evidence retrieval + LLM verification
    verifications: List[ClaimVerification] = []
    for claim in claims:
        # Retrieve the most relevant passage for THIS specific claim
        claim_docs = vs.similarity_search(claim, k=2)
        claim_evidence = format_context(claim_docs, max_chars=800)

        verdict, verify_ms = verify_claim_llm(claim, claim_evidence, llm, config)
        audit_calls += 1

        cv = ClaimVerification(
            claim=claim, verdict=verdict,
            evidence=claim_evidence[:200], llm_ms=verify_ms,
        )
        verifications.append(cv)
        logger.debug("  Claim: '%s...' -> %s", claim[:50], verdict)

    # Stage 4: Aggregate
    score         = compute_faithfulness_score(verifications)
    supported     = sum(1 for cv in verifications if cv.is_supported)
    unsupported   = sum(1 for cv in verifications if not cv.is_supported and not cv.is_contradicted)
    contradicted  = sum(1 for cv in verifications if cv.is_contradicted)
    audit_ms      = (time.perf_counter() - t0) * 1000

    report = FaithfulnessReport(
        faithfulness_score=score,
        claims=verifications,
        supported_count=supported,
        unsupported_count=unsupported,
        contradicted_count=contradicted,
        final_answer=answer,
        audit_llm_calls=audit_calls,
        audit_ms=audit_ms,
    )
    logger.info(
        "FActScore audit: score=%.3f (%d supported / %d total, %d contradicted)",
        score, supported, len(verifications), contradicted,
    )
    return report


def run_citation_audit(
    answer: str, docs: List[Document],
    llm: AzureChatOpenAI, vs: FAISS, config: FaithfulRAGConfig,
) -> List[CitationVerification]:
    """
    Citation accuracy audit for Config 4:
        1. Extract all [Source: paper_name, p.N] citations from the answer.
        2. For each citation: find the cited passage in the retrieved docs.
        3. Verify: does the cited passage actually support the claim?

    Citation accuracy is DISTINCT from answer faithfulness:
        - Faithfulness asks: "Is this claim supported by ANY passage?"
        - Citation accuracy asks: "Does the SPECIFICALLY CITED passage support it?"
    A faithfully grounded claim can have an inaccurate citation if the model
    cites the wrong passage number.
    """
    citation_pairs   = extract_inline_citations(answer)
    audit_results:   List[CitationVerification] = []
    audit_calls      = 0

    # Build a lookup: paper_name + page -> passage text
    passage_lookup: Dict[str, str] = {}
    for doc in docs:
        paper = doc.metadata.get("paper_name", "Unknown")
        page  = str(doc.metadata.get("page", "?"))
        key   = f"{paper.lower()}_p{page}"
        passage_lookup[key] = doc.page_content[:600]

    for sentence, citation_ref in citation_pairs:
        # Try to find the cited passage
        cited_passage = ""
        for key, text in passage_lookup.items():
            # Normalize citation ref and look for a key match
            norm_ref = citation_ref.lower().replace(" ", "_").replace(",", "").replace(".", "_")
            if any(part in key for part in norm_ref.split("_") if len(part) > 3):
                cited_passage = text
                break

        # If passage found: verify that it supports the claim
        is_accurate = False
        llm_ms      = 0.0
        if cited_passage:
            prompt  = config.P_VERIFY_CITATION.format(
                claim=sentence, passage=cited_passage[:600]
            )
            t0      = time.perf_counter()
            raw     = llm_call(prompt, llm)
            llm_ms  = (time.perf_counter() - t0) * 1000
            audit_calls += 1
            is_accurate = raw.strip().upper().startswith("YES")
        else:
            # Citation references a passage not in the retrieved set
            is_accurate = False

        audit_results.append(CitationVerification(
            sentence=sentence, citation_ref=citation_ref,
            cited_passage=cited_passage[:200], is_accurate=is_accurate,
            llm_ms=llm_ms,
        ))

    return audit_results


In [22]:


# ===========================================================================
# SECTION 6 -- FIVE FAITHFULNESS-OPTIMIZED CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Unchecked Baseline (standard RAG, no faithfulness measurement)
# ---------------------------------------------------------------------------

def run_config1_unchecked_baseline(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm: AzureChatOpenAI, config: FaithfulRAGConfig,
) -> FaithfulRAGTrace:
    """
    Configuration 1: Unchecked RAG Baseline.

    Standard retrieve -> generate pipeline with no faithfulness measurement
    or enforcement. This is the CONTROL CONDITION against which all other
    configurations are compared.

    The baseline uses the standard generation prompt which instructs the
    model to use "ONLY the information in the context below" -- but this
    instruction is advisory, not enforced. The model can and does deviate:
        - Injecting background knowledge not in the retrieved chunks.
        - Paraphrasing in ways that introduce subtle inaccuracies.
        - Generating confident-sounding numbers that are close but not exact.
        - Blending information from multiple chunks in ways that distort meaning.

    This is the failure mode documented in the Vectara HHEM leaderboard:
    modern LLMs have 3-27% hallucination rates on RAG tasks even when told
    to use only the provided context.

    LLM calls: 1 (generate only).
    """
    trace = FaithfulRAGTrace(question=question, strategy="Config1_Unchecked_Baseline")
    t0    = time.perf_counter()

    docs, ret_ms   = retrieve_hybrid(question, vs, bm25, config)
    context        = format_context(docs, config.MAX_CONTEXT_CHARS)
    trace.context_str  = context
    trace.retrieval_ms = ret_ms

    t_gen         = time.perf_counter()
    prompt        = config.P_GENERATE_STANDARD.format(context=context, question=question)
    answer        = llm_call(prompt, llm)
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000

    trace.raw_answer      = answer
    trace.final_answer    = answer
    trace.generation_llm  = 1
    trace.total_llm_calls = 1
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    logger.info("Config1: generation complete (%d chars, no faithfulness check).",
                len(answer))
    return trace


In [23]:

# ---------------------------------------------------------------------------
# CONFIG 2: FActScore Audit (measure faithfulness, do not modify answer)
# ---------------------------------------------------------------------------

def run_config2_factscore_audit(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm: AzureChatOpenAI, config: FaithfulRAGConfig,
) -> FaithfulRAGTrace:
    """
    Configuration 2: FActScore-style Faithfulness Audit.

    Generates the answer (same as Config 1), then runs the FActScore atomic
    claim decomposition + per-claim NLI verification pipeline.

    This config MEASURES faithfulness but does NOT modify the answer.
    It answers the diagnostic question: "What is the hallucination rate of
    this RAG pipeline?" without changing user-visible behavior.

    Use this config for:
        - Baseline evaluation before deploying faithfulness enforcement.
        - Monitoring faithfulness drift in production over time.
        - Identifying which question types trigger the most hallucination.
        - Selecting between candidate LLMs or retrieval strategies.

    LLM calls: 1 (generate) + 1 (decompose) + N_claims (verify) = 3-15 typically.
    """
    trace = FaithfulRAGTrace(question=question, strategy="Config2_FActScore_Audit")
    t0    = time.perf_counter()

    # Generation (same as Config 1)
    docs, ret_ms   = retrieve_hybrid(question, vs, bm25, config)
    context        = format_context(docs, config.MAX_CONTEXT_CHARS)
    trace.context_str  = context
    trace.retrieval_ms = ret_ms

    t_gen   = time.perf_counter()
    prompt  = config.P_GENERATE_STANDARD.format(context=context, question=question)
    answer  = llm_call(prompt, llm)
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.raw_answer = answer
    trace.generation_llm = 1

    # FActScore audit (measure only, no modification)
    report = run_factscore_audit(answer, context, docs, llm, vs, config)
    report.final_answer = answer  # unchanged

    trace.faithfulness    = report
    trace.final_answer    = answer
    trace.audit_ms        = report.audit_ms
    trace.total_llm_calls = 1 + report.audit_llm_calls
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    logger.info("Config2: faithfulness=%.3f (no regeneration).",
                report.faithfulness_score)
    return trace


In [24]:

# ---------------------------------------------------------------------------
# CONFIG 3: RAGAS Faithfulness + Citation Injection
# ---------------------------------------------------------------------------

def run_config3_ragas_with_citations(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm: AzureChatOpenAI, config: FaithfulRAGConfig,
) -> FaithfulRAGTrace:
    """
    Configuration 3: RAGAS-style Faithfulness with Citation Injection.

    Generates the answer with a CITATION-ENFORCING prompt, then runs:
        (a) RAGAS-style faithfulness audit (decompose + verify all claims).
        (b) Citation accuracy audit (verify each inline [Source: ...] reference).

    The key difference from Config 2:
        - Generation uses P_GENERATE_WITH_CITATIONS which requires the model
          to append [Source: paper_name, p.N] after every factual claim.
        - This makes the model's evidence trail EXPLICIT and auditable.
        - Citation accuracy (does [Source: X, p.N] actually support the claim?)
          is measured separately from answer faithfulness.

    The RAGAS faithfulness formula:
        faithfulness = |statements that can be inferred from context|
                       / |total statements|

    This matches the FActScore formula but uses LLM-based verification in
    both stages (decomposition AND verification), whereas FActScore originally
    uses an NLI model for verification. The LLM-based approach per FaithBench
    2025 outperforms NLI on challenging real-world hallucinations.

    LLM calls: 1 (generate) + 1 (decompose) + N_claims (verify) + N_cit (citation)
               = 3-20 depending on answer length.
    """
    trace = FaithfulRAGTrace(
        question=question, strategy="Config3_RAGAS_Citations"
    )
    t0 = time.perf_counter()

    docs, ret_ms   = retrieve_hybrid(question, vs, bm25, config)
    context        = format_context(docs, config.MAX_CONTEXT_CHARS)
    trace.context_str  = context
    trace.retrieval_ms = ret_ms

    # Citation-enforcing generation
    t_gen   = time.perf_counter()
    prompt  = config.P_GENERATE_WITH_CITATIONS.format(
        context=context, question=question
    )
    answer  = llm_call(prompt, llm)
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.raw_answer = answer
    trace.generation_llm = 1

    # FActScore / RAGAS audit
    report  = run_factscore_audit(answer, context, docs, llm, vs, config)

    # Citation accuracy audit
    cit_verifications = run_citation_audit(answer, docs, llm, vs, config)
    cit_audit_calls   = len(cit_verifications)
    accurate_cits     = sum(1 for cv in cit_verifications if cv.is_accurate)
    logger.info(
        "Config3: %d citations found, %d accurate.",
        len(cit_verifications), accurate_cits,
    )

    report.citations         = cit_verifications
    report.final_answer      = answer
    report.audit_llm_calls  += cit_audit_calls

    trace.faithfulness    = report
    trace.final_answer    = answer
    trace.audit_ms        = report.audit_ms
    trace.total_llm_calls = 1 + report.audit_llm_calls
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    logger.info("Config3: faithfulness=%.3f, citation_accuracy=%d/%d.",
                report.faithfulness_score, accurate_cits, len(cit_verifications))
    return trace


In [25]:

# ---------------------------------------------------------------------------
# CONFIG 4: Citation-Grounded RAG (enforce + verify citations)
# ---------------------------------------------------------------------------

def run_config4_citation_grounded(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm: AzureChatOpenAI, config: FaithfulRAGConfig,
) -> FaithfulRAGTrace:
    """
    Configuration 4: Citation-Grounded RAG.

    Extends Config 3 by adding a CITATION ACCURACY GATE: if citation accuracy
    is below 0.7 (i.e., more than 30% of citations are inaccurate), the answer
    is regenerated with a stricter prompt that includes only the specific
    passages that were retrieved, and cites them by their actual position
    in the context.

    This addresses the "citation hallucination" problem: the model may generate
    a correct answer but cite the wrong paper or page number. FACTUM
    (arXiv 2601.05866) identifies this as a significant trust problem:
    even experts reading a document will trust a citation without verifying it,
    and a wrong citation makes a correct claim appear to have stronger or
    different evidence than it actually does.

    The two-stage faithfulness pipeline here:
        Stage 1: Answer faithfulness (is each claim supported by any passage?)
        Stage 2: Citation accuracy (does the cited passage support the claim?)

    A production-grade RAG system needs BOTH to pass to be fully trusted.

    LLM calls: 1-2 (generate + optional regen) + 1 (decompose) + N (verify)
               + N_cit (citation check) = 4-25.
    """
    trace = FaithfulRAGTrace(
        question=question, strategy="Config4_Citation_Grounded"
    )
    t0 = time.perf_counter()

    docs, ret_ms   = retrieve_hybrid(question, vs, bm25, config)
    context        = format_context(docs, config.MAX_CONTEXT_CHARS)
    trace.context_str  = context
    trace.retrieval_ms = ret_ms

    # Initial citation-enforced generation
    t_gen   = time.perf_counter()
    prompt  = config.P_GENERATE_WITH_CITATIONS.format(
        context=context, question=question
    )
    answer  = llm_call(prompt, llm)
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.raw_answer  = answer
    trace.generation_llm = 1
    total_calls = 1

    # FActScore audit
    report = run_factscore_audit(answer, context, docs, llm, vs, config)
    total_calls += report.audit_llm_calls

    # Citation audit
    cit_verifications = run_citation_audit(answer, docs, llm, vs, config)
    cit_audit_calls   = len(cit_verifications)
    total_calls      += cit_audit_calls
    accurate_cits     = sum(1 for cv in cit_verifications if cv.is_accurate)
    cit_accuracy      = accurate_cits / max(len(cit_verifications), 1)
    report.citations  = cit_verifications

    # Citation accuracy gate: if below 0.7, regenerate with strict prompt
    CITATION_ACCURACY_THRESHOLD = 0.70
    regenerated = False
    if cit_verifications and cit_accuracy < CITATION_ACCURACY_THRESHOLD:
        logger.info(
            "Config4: citation accuracy %.2f < %.2f threshold. Regenerating...",
            cit_accuracy, CITATION_ACCURACY_THRESHOLD,
        )
        t_regen = time.perf_counter()
        strict_prompt = config.P_REGENERATE_STRICT.format(
            context=context, question=question
        )
        answer         = llm_call(strict_prompt, llm)
        trace.generation_ms += (time.perf_counter() - t_regen) * 1000
        total_calls   += 1
        regenerated    = True

        # Re-run citation audit on regenerated answer
        cit_verifications_regen = run_citation_audit(answer, docs, llm, vs, config)
        total_calls            += len(cit_verifications_regen)
        report.citations        = cit_verifications_regen

    report.regenerated   = regenerated
    report.final_answer  = answer

    trace.faithfulness    = report
    trace.final_answer    = answer
    trace.audit_ms        = report.audit_ms
    trace.total_llm_calls = total_calls
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    logger.info(
        "Config4: faithfulness=%.3f, cit_accuracy=%.2f, regenerated=%s.",
        report.faithfulness_score, cit_accuracy, regenerated,
    )
    return trace



In [26]:

# ---------------------------------------------------------------------------
# CONFIG 5: Faithful-RAG with Targeted Regeneration [BEST]
# ---------------------------------------------------------------------------

def run_config5_faithful_rag_with_regeneration(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm: AzureChatOpenAI, config: FaithfulRAGConfig,
) -> FaithfulRAGTrace:
    """
    Configuration 5: Faithful-RAG with Targeted Regeneration [BEST].

    The full faithfulness enforcement pipeline:
        Step 1: Generate answer with citation requirement.
        Step 2: Decompose into atomic claims (FActScore Stage 1).
        Step 3: Verify each claim against retrieved evidence (FActScore Stage 2).
        Step 4: If faithfulness_score >= FAITHFULNESS_THRESHOLD: answer is accepted.
        Step 5a: If faithfulness_score < threshold (full regeneration path):
                 Regenerate the ENTIRE answer with the strictest possible prompt
                 (P_REGENERATE_STRICT) and re-audit.
        Step 5b: If faithfulness_score >= threshold but some claims are unsupported
                 (partial regeneration path, REGENERATION_GRANULARITY="sentence"):
                 For each unsupported claim: retrieve targeted evidence for that claim,
                 regenerate only the SENTENCE containing the unsupported claim using
                 P_REGENERATE_SENTENCE, and splice the corrected sentence back.

    The partial regeneration path preserves ALL supported claims unchanged and
    only rewrites the specific sentences that contain unsupported claims. This:
        - Preserves answer coherence (supported parts are not degraded).
        - Minimizes LLM calls (only one regen call per unsupported claim sentence).
        - Provides a complete audit trail (original vs regenerated sentences logged).

    Self-RAG correspondence:
        This pipeline implements the ISSUP (Is Supported) critique token from
        Self-RAG (Asai et al. 2023) without requiring a fine-tuned model.
        The LLM-as-judge replaces the self-critique token, and the targeted
        regeneration replaces the model's "revise based on ISSUP=0" behavior.

    LLM calls: 1 (gen) + 1 (decompose) + N (verify) + {0..K} (sentence regen)
               + optional 1 (full regen) + optional 1 (regen decompose)
               = 3-25 total.
    """
    trace = FaithfulRAGTrace(
        question=question,
        strategy="Config5_Faithful_RAG_Regeneration [BEST]",
    )
    t0 = time.perf_counter()

    # Step 1: Retrieve + generate with citation enforcement
    docs, ret_ms   = retrieve_hybrid(question, vs, bm25, config)
    context        = format_context(docs, config.MAX_CONTEXT_CHARS)
    trace.context_str  = context
    trace.retrieval_ms = ret_ms

    t_gen   = time.perf_counter()
    prompt  = config.P_GENERATE_WITH_CITATIONS.format(
        context=context, question=question
    )
    answer  = llm_call(prompt, llm)
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.raw_answer = answer
    total_calls      = 1

    # Step 2-3: FActScore audit
    report      = run_factscore_audit(answer, context, docs, llm, vs, config)
    total_calls += report.audit_llm_calls

    logger.info(
        "Config5: initial faithfulness=%.3f (threshold=%.2f).",
        report.faithfulness_score, config.FAITHFULNESS_THRESHOLD,
    )

    # Step 4-5: Regeneration gate
    regenerated = False

    if report.faithfulness_score < config.FAITHFULNESS_THRESHOLD:
        # FULL REGENERATION PATH: too many unsupported claims
        logger.info("Config5: below threshold -> FULL regeneration.")
        t_regen = time.perf_counter()
        strict_prompt   = config.P_REGENERATE_STRICT.format(
            context=context, question=question
        )
        answer_regen    = llm_call(strict_prompt, llm)
        trace.generation_ms += (time.perf_counter() - t_regen) * 1000
        total_calls    += 1

        # Re-audit regenerated answer
        report_regen    = run_factscore_audit(
            answer_regen, context, docs, llm, vs, config
        )
        total_calls    += report_regen.audit_llm_calls

        # Accept regenerated answer (should have higher faithfulness)
        report           = report_regen
        answer           = answer_regen
        report.regenerated = True
        regenerated      = True
        logger.info(
            "Config5: FULL regeneration complete. New faithfulness=%.3f.",
            report.faithfulness_score,
        )

    elif any(not cv.is_supported for cv in report.claims):
        # PARTIAL REGENERATION PATH: above threshold but some claims are unsupported
        # Fix ONLY the sentences containing unsupported claims.
        unsupported_cvs = [cv for cv in report.claims if not cv.is_supported]
        logger.info(
            "Config5: %d unsupported claims -> targeted sentence regeneration.",
            len(unsupported_cvs),
        )

        for cv in unsupported_cvs:
            # Retrieve targeted evidence for this specific claim
            claim_docs = vs.similarity_search(cv.claim, k=1)
            if not claim_docs:
                continue
            supporting_passage = claim_docs[0].page_content[:400]

            # Find the sentence in the answer that contains this claim
            # (match by the first 30 chars of the claim -- robust to minor rewordings)
            claim_snippet   = cv.claim[:30].lower()
            answer_sentences = re.split(r"(?<=[.!?])\s+", answer)
            target_sent_idx = None
            for i, sent in enumerate(answer_sentences):
                if claim_snippet in sent.lower():
                    target_sent_idx = i
                    break

            if target_sent_idx is None:
                continue

            original_sentence = answer_sentences[target_sent_idx]

            # Regenerate ONLY this sentence
            regen_prompt    = config.P_REGENERATE_SENTENCE.format(
                original_sentence=original_sentence,
                supporting_passage=supporting_passage,
                unsupported_claim=cv.claim,
            )
            t_regen         = time.perf_counter()
            fixed_sentence  = llm_call(regen_prompt, llm)
            trace.generation_ms += (time.perf_counter() - t_regen) * 1000
            total_calls    += 1

            # Splice the fixed sentence back into the answer
            answer_sentences[target_sent_idx] = fixed_sentence
            answer = " ".join(answer_sentences)

            logger.info(
                "Config5: replaced sentence [%s...] -> [%s...]",
                original_sentence[:40], fixed_sentence[:40],
            )

        # Re-run faithfulness audit on the patched answer
        report_final    = run_factscore_audit(answer, context, docs, llm, vs, config)
        total_calls    += report_final.audit_llm_calls
        report          = report_final
        report.regenerated = True
        regenerated     = True
        logger.info(
            "Config5: partial regeneration complete. Final faithfulness=%.3f.",
            report.faithfulness_score,
        )

    # Finalize
    report.final_answer  = answer
    report.regenerated   = regenerated

    trace.faithfulness    = report
    trace.final_answer    = answer
    trace.audit_ms        = report.audit_ms
    trace.total_llm_calls = total_calls
    trace.total_ms        = (time.perf_counter() - t0) * 1000
    logger.info(
        "Config5 complete: final faithfulness=%.3f, regenerated=%s.",
        report.faithfulness_score, regenerated,
    )
    return trace



In [27]:

# ===========================================================================
# SECTION 7 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question: str, vs: FAISS, bm25: BM25Retriever,
    llm: AzureChatOpenAI, config: FaithfulRAGConfig,
) -> Dict[str, FaithfulRAGTrace]:
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Unchecked_Baseline":      lambda q: run_config1_unchecked_baseline(
            q, vs, bm25, llm, config),
        "Config2_FActScore_Audit":         lambda q: run_config2_factscore_audit(
            q, vs, bm25, llm, config),
        "Config3_RAGAS_Citations":         lambda q: run_config3_ragas_with_citations(
            q, vs, bm25, llm, config),
        "Config4_Citation_Grounded":       lambda q: run_config4_citation_grounded(
            q, vs, bm25, llm, config),
        "Config5_FaithfulRAG_Regen [BEST]":lambda q: run_config5_faithful_rag_with_regeneration(
            q, vs, bm25, llm, config),
    }

    results: Dict[str, FaithfulRAGTrace] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            tr = fn(question)
            tr.print_trace()
            results[label] = tr
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("FAITHFULNESS-OPTIMIZED RAG COMPARISON SUMMARY")
    print(f"{'Config':<44} {'Faith':>6} {'Claims':>6} {'LLM':>5} "
          f"{'Regen':>5} {'Total_ms':>9}")
    print("-" * 78)
    for lbl, tr in results.items():
        f_score  = f"{tr.faithfulness.faithfulness_score:.3f}" if tr.faithfulness else "N/A"
        n_claims = len(tr.faithfulness.claims) if tr.faithfulness else 0
        regen    = "YES" if (tr.faithfulness and tr.faithfulness.regenerated) else "NO"
        print(f"{lbl:<44} {f_score:>6} {n_claims:>6} {tr.total_llm_calls:>5} "
              f"{regen:>5} {tr.total_ms:>9.0f}")
    print("=" * 78 + "\n")
    return results


In [28]:
 """
    End-to-end Faithfulness-Optimized RAG pipeline orchestrator.

    INDEXING (run once, cached): ~2 min.
        - Download 3 arXiv PDFs.
        - Chunk -> BGE-large embed -> FAISS index.
        - Build BM25 index in memory.

    LLM CALL BUDGET PER QUESTION:
        Config 1:  1  call  (no audit).
        Config 2:  1 + 1 + N_claims calls (generate + decompose + verify).
        Config 3:  1 + 1 + N_claims + N_citations calls.
        Config 4:  1-2 + 1 + N_claims + N_citations calls (+ optional regen).
        Config 5:  1-3 + 1-2 + N_claims + N_sentence_regens calls.

    TYPICAL TOTAL:
        Config 1:  1 call.
        Config 2:  5-14 calls.
        Config 3:  6-20 calls.
        Config 4:  7-22 calls.
        Config 5:  8-25 calls (most expensive, highest faithfulness guarantee).

    DEMO QUESTIONS: chosen to expose specific failure modes of unchecked RAG.
        - Numeric precision: model may round or misquote specific model dimensions.
        - Multi-paper synthesis: model may blend facts from two papers incorrectly.
        - Architectural detail: model may add background knowledge not in the context.
    """

'\n   End-to-end Faithfulness-Optimized RAG pipeline orchestrator.\n\n   INDEXING (run once, cached): ~2 min.\n       - Download 3 arXiv PDFs.\n       - Chunk -> BGE-large embed -> FAISS index.\n       - Build BM25 index in memory.\n\n   LLM CALL BUDGET PER QUESTION:\n       Config 1:  1  call  (no audit).\n       Config 2:  1 + 1 + N_claims calls (generate + decompose + verify).\n       Config 3:  1 + 1 + N_claims + N_citations calls.\n       Config 4:  1-2 + 1 + N_claims + N_citations calls (+ optional regen).\n       Config 5:  1-3 + 1-2 + N_claims + N_sentence_regens calls.\n\n   TYPICAL TOTAL:\n       Config 1:  1 call.\n       Config 2:  5-14 calls.\n       Config 3:  6-20 calls.\n       Config 4:  7-22 calls.\n       Config 5:  8-25 calls (most expensive, highest faithfulness guarantee).\n\n   DEMO QUESTIONS: chosen to expose specific failure modes of unchecked RAG.\n       - Numeric precision: model may round or misquote specific model dimensions.\n       - Multi-paper synthesi

In [32]:
config = FaithfulRAGConfig()


In [33]:
pdf_paths  = download_pdfs(config)
chunks     = load_and_chunk(pdf_paths, config)
embeddings = build_bge_embeddings(config)
vs         = build_faiss(chunks, embeddings, config)
bm25       = build_bm25(chunks)
llm        = build_azure_llm(config)


2026-05-27 22:23:11 | INFO     | faithfulness_rag |   attention_is_all_you_need.pdf -> 104 chunks
2026-05-27 22:23:13 | INFO     | faithfulness_rag |   bert_pretraining_transformers.pdf -> 173 chunks
2026-05-27 22:23:14 | INFO     | faithfulness_rag |   rag_knowledge_intensive_nlp.pdf -> 181 chunks
2026-05-27 22:23:14 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


C:\Users\dhanu\AppData\Local\Temp\ipykernel_22664\1111385766.py:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


2026-05-27 22:23:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:23:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-27 22:23:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:23:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-27 22:23:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


2026-05-27 22:23:16 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-27 22:23:16 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-27 22:23:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:23:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/README.md "HTTP/1.1 200 OK"
2026-05-27 22:23:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:23:17 | INFO     | httpx | HTTP 

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2184.17it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-27 22:23:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:23:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-27 22:23:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 22:23:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-27 22:23:20 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-27 22:23:20 | INFO     | httpx |

In [29]:
demo_questions = [
    # Numeric: exposes rounding/misquoting hallucinations
    "What are the exact hidden size, number of heads, and feed-forward size of the BERT-Base model?",

    # Multi-paper synthesis: tests cross-paper attribution
    "How does the position encoding used in the original Transformer compare to how BERT handles positional information?",

    # Architectural detail: tests parametric knowledge injection
    "What is the purpose of the layer normalization in the Transformer encoder and where exactly is it applied?",

    # Factual precision: easy question with specific numeric answer
    "How many encoder and decoder layers does the Transformer base model have?",

    # Claim verification stress test: answer should contain ~5-8 claims
    "What retrieval mechanism does the RAG paper use, and what knowledge source does it retrieve from?",
]

In [34]:
for question in demo_questions[:2]:    # run 2 for demo (5 full is expensive)
    run_all_configs(question, vs, bm25, llm, config)

logger.info("=== Faithfulness-Optimized RAG Pipeline Demo Complete ===")



##############################################################################
QUERY: What are the exact hidden size, number of heads, and feed-forward size of the BERT-Base model?
##############################################################################

RUNNING: Config1_Unchecked_Baseline
2026-05-27 22:23:31 | INFO     | httpx | HTTP Request: POST https://idkopenai.services.ai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-27 22:23:31 | INFO     | faithfulness_rag | Config1: generation complete (61 chars, no faithfulness check).

TRACE: Config1_Unchecked_Baseline
Query: What are the exact hidden size, number of heads, and feed-forward size o
  LLM calls:   1  | Retrieval: 1762ms  | Gen: 2433ms  | Audit: 0ms  | Total: 4195ms

  FINAL ANSWER:
  The provided documents do not contain sufficient information.


RUNNING: Config2_FActScore_Audit
2026-05-27 22:23:33 | INFO     | httpx | HTTP Request: POST https://idkopenai.s

In [4]:
# Extra faithfulness stress-test examples
extra_demo_questions = [
    "What exact evidence supports the claim that Transformer uses scaled dot-product attention, and what is the scaling term?",
    "Does the RAG paper retrieve from parametric memory or an external non-parametric index, and where is this stated?",
    "What are the precise BERT-Base architecture numbers (layers, hidden size, attention heads), with citation-backed claims?",
]

RUN_EXTRA_DEMOS = False
if RUN_EXTRA_DEMOS:
    for question in extra_demo_questions[:2]:
        run_all_configs(question, vs, bm25, llm, config)
else:
    print("Set RUN_EXTRA_DEMOS = True to execute extra Faithfulness-Optimized examples.")

Set RUN_EXTRA_DEMOS = True to execute extra Faithfulness-Optimized examples.
